In [1]:
%cd /media/ecampbell/D/code/sst

# %%
import os
import torch
from datasets import load_dataset, DatasetDict, Audio, Dataset
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    BitsAndBytesConfig
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, PeftModel, PeftConfig, TaskType
import evaluate
from tqdm.auto import tqdm

# %%
# Define constants
version = "tiny"
MODEL_NAME = f"openai/whisper-{version}"
DATA_DIR = "data_source/data_readspeech_am/data"
LANGUAGE = "am" # Amharic ISO code
TASK = "transcribe"


/media/ecampbell/D/code/sst


In [2]:
# Load dataset
# Custom loader to handle wav and txt files
def load_custom_dataset(data_dir, max_num_samples=None):
    data = {}
    for split in ["train", "test"]:
        split_dir = os.path.join(data_dir, split)
        if not os.path.isdir(split_dir):
            continue

        # Index wav files to handle potential subdirectories or missing extensions
        wav_files = {}
        for root, _, files in tqdm(os.walk(split_dir), desc=f"Loading {split} audiosamples..."):
            for file in files:
                if file.endswith(".wav"):
                    abs_path = os.path.join(root, file)
                    wav_files[file] = abs_path
                    wav_files[os.path.splitext(file)[0]] = abs_path

        audio_paths = []
        transcriptions = []

        text_path = os.path.join(split_dir, "text")
        if os.path.exists(text_path):
            with open(text_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split(maxsplit=1)
                    if len(parts) == 2:
                        audio_name, text = parts
                        if audio_name in wav_files:
                            audio_paths.append(wav_files[audio_name])
                            transcriptions.append(text)
        if max_num_samples:
            audio_paths = audio_paths[:max_num_samples]
            transcriptions = transcriptions[:max_num_samples]
        if audio_paths:
            data[split] = Dataset.from_dict({"audio": audio_paths, "text": transcriptions})

    return DatasetDict(data)

dataset = load_custom_dataset(DATA_DIR, max_num_samples=1_000)
# print(dataset)

Loading test audiosamples...: 0it [00:00, ?it/s]

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'text'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['audio', 'text'],
        num_rows: 359
    })
})

In [4]:
# Preprocessing
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)


In [5]:

# Resample audio to 16kHz
# dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [6]:
# import os
# num_cpu_to_use = 1#round(0.5 * os.cpu_count())
# # %%
# def prepare_dataset(batch):
#     # load and resample audio data from 48 to 16kHz
#     audio = batch["audio"]
#
#     # compute log-Mel input features from input audio array
#     batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
#
#     # encode target text to label ids
#     batch["labels"] = tokenizer(batch["text"]).input_ids # Assuming 'text' column exists
#     return batch
#
# # %%
# encoded_dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names["train"], num_proc=num_cpu_to_use)

In [5]:
import os
from datasets import load_from_disk, Audio

def prepare_dataset_batched(batch):
    # Because we used cast_column, these arrays are already guaranteed to be 16kHz
    audio_arrays = [audio["array"] for audio in batch["audio"]]

    # Process the entire batch of audio arrays at once
    # The feature extractor will automatically pad them to 30 seconds (for Whisper)
    batch["input_features"] = feature_extractor(
        audio_arrays,
        sampling_rate=16000
    ).input_features

    # Process the entire batch of text at once
    batch["labels"] = tokenizer(batch["text"]).input_ids

    return batch

save_path = "whisper_processed_data"

if os.path.exists(save_path):
    print("Found pre-processed dataset on disk. Loading instantly...")
    encoded_dataset = load_from_disk(save_path)

else:
    print("Processed dataset not found. Starting feature extraction...")

    # 1. Cast audio to 16kHz
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

    # 2. Map the dataset (using your batched function)
    encoded_dataset = dataset.map(
        prepare_dataset_batched,
        remove_columns=dataset.column_names["train"],
        batched=True,
        batch_size=100,
        num_proc=1#max(1, os.cpu_count() - 2)
    )

    # 3. Save it so we never have to do this again
    print("Extraction complete. Saving to disk...")
    encoded_dataset.save_to_disk(save_path)
    print("Done!")

# Now you are ready to pass encoded_dataset directly to your Trainer!

Found pre-processed dataset on disk. Loading instantly...


In [6]:

# Model with Quantization (QLoRA)
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    # attn_implementation="flash_attention_2", # <-- 1. Enable Flash Attention 2
    # torch_dtype=torch.float16                # <-- 2. Mandatory for FA2 compute)
)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
model.generation_config.suppress_tokens = []

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

# Apply LoRA
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
model.config.use_cache = False
model.enable_input_require_grads()


trainable params: 589,824 || all params: 38,350,464 || trainable%: 1.5380


In [7]:
# Data Collator
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [8]:
# Metrics
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}


In [11]:

# Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=f"./whisper-{version}-finetuned",  # change to a repo name of your choice
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-3, # Higher LR for LoRA
    warmup_steps=50,
    # max_steps=2, # Increased steps slightly
    num_train_epochs=50,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225, #TODO review this lenght to our application
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
    remove_unused_columns=False,
    label_names=["labels"],
)


In [12]:

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics)


In [13]:

trainer.train()


TrainOutput(global_step=3150, training_loss=0.22066858292926872, metrics={'train_runtime': 28466.5859, 'train_samples_per_second': 1.756, 'train_steps_per_second': 0.111, 'total_flos': 1.273411584e+18, 'train_loss': 0.22066858292926872, 'epoch': 50.0})

In [14]:

# Evaluation
trainer.evaluate()

{'eval_loss': 0.9034744501113892,
 'eval_wer': 0.8315331291310656,
 'eval_runtime': 294.4952,
 'eval_samples_per_second': 1.219,
 'eval_steps_per_second': 0.153,
 'epoch': 50.0}

In [15]:
# Save the model
trainer.save_model()
processor.save_pretrained(f"./whisper-{version}-finetuned")


['./whisper-tiny-finetuned/processor_config.json']

In [9]:
# Inference
from transformers import pipeline
from peft import PeftModel, PeftConfig

device = "cuda" if torch.cuda.is_available() else "cpu"

# For inference with PEFT, we can use the model object directly if it's in memory
# or load it. Since we just trained it, 'model' is available.
# However, pipeline might struggle with the PEFT wrapper directly in some versions.
# But let's try using the in-memory model which is safest.

pipe = pipeline(
    "automatic-speech-recognition",
    model=f"./whisper-{version}-finetuned",#model,
    tokenizer=tokenizer,
    feature_extractor=feature_extractor,
    # device=device, # device_map="auto" handles device placement
)

# Pick a sample
sample = dataset["test"][0]
audio = sample["audio"]

# Transcribe
result = pipe(audio, generate_kwargs={"language": LANGUAGE, "task": TASK, "forced_decoder_ids": processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)})

print(f"Reference: {sample['text']}")
print(f"Prediction: {result['text']}")


Reference: የ ኢንተርኔት አገልግሎት ንም በ ተመለከተ በ ክልሎች በ ዞኖች ና በ አዲስ አበባ በ ተለያዩ ቦታዎች የ አገልግሎት ማእከሎ ችን ለ ማቋቋም መታቀዱ ን አብራር ተዋል
Prediction: የ ኢንተልነት አገልግ ሎተን በ ተመለካተሁ ተላዎች በ ዛኖች ና በ አዲሳ በ በ ተለያዩ ቦታዎች የ አገል ግሎት መአክሎች ን ለማቋቂ ም ምታቅድ ን ም አብራር ተዋል


In [10]:
# Compute WER (wrap strings in lists)
sample_wer = wer_metric.compute(predictions=[result["text"]], references=[sample["text"]])

# Multiply by 100 to display it as a percentage
print(f"Sample WER: {sample_wer * 100:.2f}%")

Sample WER: 62.96%
